# Train the student TotalSegmentator using as basis the segmentations of the original TotalSegmentator

## Decisions:
* Use the 1.5×1.5×1.5 mm voxels Professor TotalSegmentator
* Change the class mapping to CLASS_MAPPING = {"organs": 1, "cardiac": 2, "muscles": 3, "bones": 4, ribs": 5, "vertebrae": 6}
* 

### Convert to Nifti

In [ ]:
# Convert mha to nifti
import os
from os import listdir
from os.path import join, isdir
import SimpleITK as sitk

## Most cases
root_dataset = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/synthRAD2025_Task1_Train/Task1"
root_nii_dataset = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/nifty_dataset/Task1"
for region in listdir(root_dataset):
    region_path = join(root_dataset, region)
    if not isdir(region_path):
        continue
    for case in listdir(region_path):
        mha_path = join(region_path, case, 'ct.mha')
        nii_folder = join(root_nii_dataset, region, case)
        nii_path = join(nii_folder, 'ct.nii.gz')
        os.makedirs(nii_folder, exist_ok=True)
        img = sitk.ReadImage(mha_path)
        sitk.WriteImage(img, nii_path)

In [ ]:
# Convert mha to nifti
import os
from os import listdir
from os.path import join, isdir
import SimpleITK as sitk

## Most cases
root_dataset = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/synthRAD2025_Task1_Train_D/Task1"
root_nii_dataset = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/nifty_dataset/Task1"
for region in listdir(root_dataset):
    region_path = join(root_dataset, region)
    if not isdir(region_path):
        continue
    for case in listdir(region_path):
        mha_path = join(region_path, case, 'ct.mha')
        nii_folder = join(root_nii_dataset, region, case)
        nii_path = join(nii_folder, 'ct.nii.gz')
        os.makedirs(nii_folder, exist_ok=True)
        img = sitk.ReadImage(mha_path)
        sitk.WriteImage(img, nii_path)

### Run the TotalSegmentator

In [ ]:
import os
from os import listdir
from os.path import join, isdir
import subprocess

root_seg_path = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/Task1_HighRes_seg"
## Most cases
root_dataset = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/nifty_dataset/Task1"

for region in listdir(root_dataset):
    region_path = join(root_dataset, region)
    if not isdir(region_path):
        continue
    for case in listdir(region_path):
        ct_case_path = join(region_path, case, 'ct.nii.gz')
        seg_dir = join(root_seg_path, region, case)
        seg_path = join(seg_dir, 'pred_seg.nii.gz')
        os.makedirs(seg_dir, exist_ok=True)
        # run seg command
        # TotalSegmentator -i ct_case_path -o seg_path
        cmd = [
            "TotalSegmentator",
            "-i", ct_case_path,
            "-o", seg_path
        ]
        print(f"🧩 Running: {' '.join(cmd)}")
        subprocess.run(cmd, check=True)

In [ ]:
import os
from os import listdir
from os.path import join, isdir
import subprocess
import argparse
import nibabel as nib
import numpy as np
from glob import glob

def create_compact_label(input_dir, pattern, out_path, overwrite):
    files = sorted(glob(os.path.join(input_dir, pattern)))
    if not files:
        raise SystemExit(f"No files found in {input_dir} with pattern {pattern}")

    # Load first image to get shape, affine, header
    first_img = nib.load(files[0])
    shape = first_img.shape
    affine = first_img.affine
    header = first_img.header

    # Create output array, dtype = uint8 or int16 if many classes (here small ints)
    out = np.zeros(shape, dtype=np.uint8)

    unknown_files = []
    mismatched_shape = []

    for fn in files:
        base = os.path.basename(fn)
        name = base.split(".")[0]  # take file name without extension(s)
        # If filename has multiple dots (e.g., label.nii.gz), above still works.
        if name not in LABEL_TO_CLASS:
            unknown_files.append(base)
            print(f"WARNING: label '{name}' not found in LABEL_TO_CLASS -> skipping")
            continue
        class_id = LABEL_TO_CLASS[name]

        img = nib.load(fn)
        data = img.get_fdata()
        # If binary label masks are not 0/1, treat non-zero as label
        mask = data != 0

        if img.shape != shape:
            mismatched_shape.append(base)
            print(f"ERROR: shape mismatch for {base}: {img.shape} != {shape} -- skipping")
            continue

        if overwrite:
            # later file overwrites previous values (where mask is True)
            out[mask] = class_id
        else:
            # first-wins: set only where out is zero
            to_set = mask & (out == 0)
            out[to_set] = class_id


    # Save result as NIfTI (use first image's affine & header where possible)
    out_img = nib.Nifti1Image(out.astype(np.uint8), affine, header)
    nib.save(out_img, out_path)
    print(f"\nSaved combined class segmentation to: {out_path}")

    if unknown_files:
        print("\nFiles skipped because label not recognized (check names or add to LABEL_TO_CLASS):")
        for u in unknown_files:
            print("  -", u)
    if mismatched_shape:
        print("\nFiles skipped due to shape mismatch:")
        for m in mismatched_shape:
            print("  -", m)

In [ ]:
# Class mapping (fixed typo "ribs")
CLASS_MAPPING = {"organs": 1, "cardiac": 2, "muscles": 3, "bones": 4, "ribs": 5, "vertebrae": 6}

# Label -> class id mapping (from the mapping we discussed earlier)
LABEL_TO_CLASS = {
    # Organs
    "spleen": CLASS_MAPPING["organs"],
    "kidney_right": CLASS_MAPPING["organs"],
    "kidney_left": CLASS_MAPPING["organs"],
    "gallbladder": CLASS_MAPPING["organs"],
    "liver": CLASS_MAPPING["organs"],
    "stomach": CLASS_MAPPING["organs"],
    "pancreas": CLASS_MAPPING["organs"],
    "adrenal_gland_right": CLASS_MAPPING["organs"],
    "adrenal_gland_left": CLASS_MAPPING["organs"],
    "esophagus": CLASS_MAPPING["organs"],
    "trachea": CLASS_MAPPING["organs"],
    "thyroid_gland": CLASS_MAPPING["organs"],
    "small_bowel": CLASS_MAPPING["organs"],
    "duodenum": CLASS_MAPPING["organs"],
    "colon": CLASS_MAPPING["organs"],
    "urinary_bladder": CLASS_MAPPING["organs"],
    "prostate": CLASS_MAPPING["organs"],
    "kidney_cyst_left": CLASS_MAPPING["organs"],
    "kidney_cyst_right": CLASS_MAPPING["organs"],
    "brain": CLASS_MAPPING["organs"],
    "lung_lower_lobe_left": CLASS_MAPPING["organs"],
    "lung_lower_lobe_right": CLASS_MAPPING["organs"],
    "lung_middle_lobe_right" : CLASS_MAPPING["organs"],
    "lung_upper_lobe_left": CLASS_MAPPING["organs"],
    "lung_upper_lobe_right": CLASS_MAPPING["organs"],
    
    # Cardiac / vascular
    "heart": CLASS_MAPPING["cardiac"],
    "aorta": CLASS_MAPPING["cardiac"],
    "pulmonary_vein": CLASS_MAPPING["cardiac"],
    "brachiocephalic_trunk": CLASS_MAPPING["cardiac"],
    "subclavian_artery_right": CLASS_MAPPING["cardiac"],
    "subclavian_artery_left": CLASS_MAPPING["cardiac"],
    "common_carotid_artery_right": CLASS_MAPPING["cardiac"],
    "common_carotid_artery_left": CLASS_MAPPING["cardiac"],
    "brachiocephalic_vein_left": CLASS_MAPPING["cardiac"],
    "brachiocephalic_vein_right": CLASS_MAPPING["cardiac"],
    "atrial_appendage_left": CLASS_MAPPING["cardiac"],
    "superior_vena_cava": CLASS_MAPPING["cardiac"],
    "inferior_vena_cava": CLASS_MAPPING["cardiac"],
    "portal_vein_and_splenic_vein": CLASS_MAPPING["cardiac"],
    "iliac_artery_left": CLASS_MAPPING["cardiac"],
    "iliac_artery_right": CLASS_MAPPING["cardiac"],
    "iliac_vena_left": CLASS_MAPPING["cardiac"],
    "iliac_vena_right": CLASS_MAPPING["cardiac"],

    # Muscles
    "gluteus_maximus_left": CLASS_MAPPING["muscles"],
    "gluteus_maximus_right": CLASS_MAPPING["muscles"],
    "gluteus_medius_left": CLASS_MAPPING["muscles"],
    "gluteus_medius_right": CLASS_MAPPING["muscles"],
    "gluteus_minimus_left": CLASS_MAPPING["muscles"],
    "gluteus_minimus_right": CLASS_MAPPING["muscles"],
    "iliopsoas_left": CLASS_MAPPING["muscles"],
    "iliopsoas_right": CLASS_MAPPING["muscles"],
    "autochthon_left": CLASS_MAPPING["muscles"],
    "autochthon_right": CLASS_MAPPING["muscles"],

    # Bones
    "sacrum": CLASS_MAPPING["bones"],
    "humerus_left": CLASS_MAPPING["bones"],
    "humerus_right": CLASS_MAPPING["bones"],
    "scapula_left": CLASS_MAPPING["bones"],
    "scapula_right": CLASS_MAPPING["bones"],
    "clavicula_left": CLASS_MAPPING["bones"],
    "clavicula_right": CLASS_MAPPING["bones"],
    "femur_left": CLASS_MAPPING["bones"],
    "femur_right": CLASS_MAPPING["bones"],
    "hip_left": CLASS_MAPPING["bones"],
    "hip_right": CLASS_MAPPING["bones"],
    "skull": CLASS_MAPPING["bones"],
    "sternum": CLASS_MAPPING["bones"],
    "costal_cartilages": CLASS_MAPPING["bones"],

    # Ribs
    "rib_left_1": CLASS_MAPPING["ribs"],
    "rib_left_2": CLASS_MAPPING["ribs"],
    "rib_left_3": CLASS_MAPPING["ribs"],
    "rib_left_4": CLASS_MAPPING["ribs"],
    "rib_left_5": CLASS_MAPPING["ribs"],
    "rib_left_6": CLASS_MAPPING["ribs"],
    "rib_left_7": CLASS_MAPPING["ribs"],
    "rib_left_8": CLASS_MAPPING["ribs"],
    "rib_left_9": CLASS_MAPPING["ribs"],
    "rib_left_10": CLASS_MAPPING["ribs"],
    "rib_left_11": CLASS_MAPPING["ribs"],
    "rib_left_12": CLASS_MAPPING["ribs"],
    "rib_right_1": CLASS_MAPPING["ribs"],
    "rib_right_2": CLASS_MAPPING["ribs"],
    "rib_right_3": CLASS_MAPPING["ribs"],
    "rib_right_4": CLASS_MAPPING["ribs"],
    "rib_right_5": CLASS_MAPPING["ribs"],
    "rib_right_6": CLASS_MAPPING["ribs"],
    "rib_right_7": CLASS_MAPPING["ribs"],
    "rib_right_8": CLASS_MAPPING["ribs"],
    "rib_right_9": CLASS_MAPPING["ribs"],
    "rib_right_10": CLASS_MAPPING["ribs"],
    "rib_right_11": CLASS_MAPPING["ribs"],
    "rib_right_12": CLASS_MAPPING["ribs"],

    # Vertebrae
    "vertebrae_S1": CLASS_MAPPING["vertebrae"],
    "vertebrae_L5": CLASS_MAPPING["vertebrae"],
    "vertebrae_L4": CLASS_MAPPING["vertebrae"],
    "vertebrae_L3": CLASS_MAPPING["vertebrae"],
    "vertebrae_L2": CLASS_MAPPING["vertebrae"],
    "vertebrae_L1": CLASS_MAPPING["vertebrae"],
    "vertebrae_T12": CLASS_MAPPING["vertebrae"],
    "vertebrae_T11": CLASS_MAPPING["vertebrae"],
    "vertebrae_T10": CLASS_MAPPING["vertebrae"],
    "vertebrae_T9": CLASS_MAPPING["vertebrae"],
    "vertebrae_T8": CLASS_MAPPING["vertebrae"],
    "vertebrae_T7": CLASS_MAPPING["vertebrae"],
    "vertebrae_T6": CLASS_MAPPING["vertebrae"],
    "vertebrae_T5": CLASS_MAPPING["vertebrae"],
    "vertebrae_T4": CLASS_MAPPING["vertebrae"],
    "vertebrae_T3": CLASS_MAPPING["vertebrae"],
    "vertebrae_T2": CLASS_MAPPING["vertebrae"],
    "vertebrae_T1": CLASS_MAPPING["vertebrae"],
    "vertebrae_C7": CLASS_MAPPING["vertebrae"],
    "vertebrae_C6": CLASS_MAPPING["vertebrae"],
    "vertebrae_C5": CLASS_MAPPING["vertebrae"],
    "vertebrae_C4": CLASS_MAPPING["vertebrae"],
    "vertebrae_C3": CLASS_MAPPING["vertebrae"],
    "vertebrae_C2": CLASS_MAPPING["vertebrae"],
    "vertebrae_C1": CLASS_MAPPING["vertebrae"],

    # Nervous system (kept as organs category here)
    "spinal_cord": CLASS_MAPPING["organs"],
}

root_seg_path = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/nnUNet/raw_data/Task2/segs"
nnunet_folder = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/nnUNet/raw_data/Task2/labels"
pattern = "*.nii*"
overwrite = False

for file in listdir(root_seg_path):
        # Input folder with all segmentations
        file_path = join(root_seg_path, file)
        input_dir = join(file_path)
        # Output folder for compact label
        out_folder = join(nnunet_folder, file)
        os.makedirs(out_folder, exist_ok=True)
        out_path = join(out_folder, "pred_seg.nii.gz")
        if os.path.exists(out_path):
            continue
        print(f"Doing: {input_dir}")
        try:
            create_compact_label(input_dir, pattern, out_path, overwrite)
        except:
            continue


### Convert to nnUNet format

In [ ]:
import os
import shutil
from os import listdir
from os.path import join, isdir

nifti_path = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/nnUNet/raw_data/Task1/nifty_dataset"
seg_path = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/nnUNet/raw_data/Task1/segs"

imagesTr = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/nnUNet/nnUNet_raw/Dataset100_WholeBodyCT/imagesTr"
labelsTr = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/nnUNet/nnUNet_raw/Dataset100_WholeBodyCT/labelsTr"

for region in listdir(nifti_path):
    region_path = join(nifti_path, region)
    for case in listdir(region_path):
        # Origin path
        ct_case_path = join(region_path, case, "ct.nii.gz")
        seg_case_path = join(seg_path, region, case, "pred_seg.nii.gz")

        # Dest path 
        ct_dest_path = join(imagesTr, f"{case}_0000.nii.gz")
        seg_dest_path = join(labelsTr, f"{case}.nii.gz")

        # Copy files if they exist
        if os.path.exists(ct_case_path):
            shutil.copy(ct_case_path, ct_dest_path)
            print(f"Copied CT: {ct_case_path} → {ct_dest_path}")
        else:
            print(f"⚠️ Missing CT file: {ct_case_path}")

        if os.path.exists(seg_case_path):
            shutil.copy(seg_case_path, seg_dest_path)
            print(f"Copied SEG: {seg_case_path} → {seg_dest_path}")
        else:
            print(f"⚠️ Missing SEG file: {seg_case_path}")
        

       

In [ ]:
export nnUNet_raw="/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/nnUNet/nnUNet_raw"
export nnUNet_preprocessed="/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/nnUNet/nnUNet_preprocessed"
export nnUNet_results="/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/nnUNet/nnUNet_results"